# 05 Rule-Based Flags
This notebook checks each transaction for suspicious patterns using simple SQL rules. It reads from silver.transactions and the edgelist, then saves the results to gold.rule_flags.

%md
## Rule 1: Fan-out (splitting)
Count how many transactions each txId sends money TO. If a transaction sends to many others, it could be splitting money to hide it.

In [0]:
SELECT
  txId1 AS txId,
  COUNT(txId2) AS sends_to
FROM silver.elliptic_txs_edgelist
GROUP BY txId1
ORDER BY sends_to DESC
LIMIT 20

%md
## Rule 2: Fan-in (gathering)
Count how many transactions send money TO each txId. If many transactions all send to one place, it could be gathering laundered money together.

In [0]:
SELECT txId2 AS txId, COUNT(*) AS received_from
FROM silver.elliptic_txs_edgelist
GROUP BY txId2
ORDER BY received_from DESC
LIMIT 20

%md
## Rule 3: Connected to a known illicit transaction
If a transaction sends money directly to a known illicit transaction, it is also suspicious.

In [0]:
SELECT e.txId1 AS txId, t.label_name AS connected_to
FROM silver.elliptic_txs_edgelist e
JOIN silver.transactions t ON e.txId2 = t.txId
WHERE t.label_name = 'illicit'
LIMIT 20

%md
## Step 4: Save each rule as a temporary list
We save the 3 rules with names so we can combine them in the next step.

In [0]:
CREATE OR REPLACE TEMP VIEW fan_out AS
SELECT txId1 AS txId, COUNT(*) AS sends_to
FROM silver.elliptic_txs_edgelist
GROUP BY txId1

In [0]:
CREATE OR REPLACE TEMP VIEW fan_in AS
SELECT txId2 AS txId, COUNT(*) AS received_from
FROM silver.elliptic_txs_edgelist
GROUP BY txId2

In [0]:
CREATE OR REPLACE TEMP VIEW linked_illicit AS
SELECT DISTINCT txId1 AS txId
FROM silver.elliptic_txs_edgelist
WHERE txId2 IN (
  SELECT txId FROM silver.transactions WHERE label_name = 'illicit'
)

%md
## Step 5: What is normal?
We check how many transactions send to (or receive from) 1, 2, 3... others. This helps us choose a fair limit for "too many".

In [0]:
SELECT sends_to, COUNT(*) AS transactions
FROM fan_out
GROUP BY sends_to
ORDER BY sends_to
LIMIT 15

In [0]:
SELECT received_from, COUNT(*) AS transactions
FROM fan_in
GROUP BY received_from
ORDER BY received_from
LIMIT 15

%md
## Step 6: Combine the rules and give a risk score
Fan-out or fan-in of 10 or more is flagged, because over 9 in 10 transactions only link to 1 or 2 others.
Risk score: fan-out = 30 points, fan-in = 30 points, linked to illicit = 40 points (strongest sign). Maximum = 100.

In [0]:
CREATE OR REPLACE TEMP VIEW all_flags AS
SELECT
  t.txId,
  t.time_step,
  t.label_name,
  COALESCE(o.sends_to, 0) AS sends_to,
  COALESCE(i.received_from, 0) AS received_from,
  CASE WHEN o.sends_to >= 10 THEN 1 ELSE 0 END AS flag_fan_out,
  CASE WHEN i.received_from >= 10 THEN 1 ELSE 0 END AS flag_fan_in,
  CASE WHEN l.txId IS NOT NULL THEN 1 ELSE 0 END AS flag_linked_illicit
FROM silver.transactions t
LEFT JOIN fan_out o ON t.txId = o.txId
LEFT JOIN fan_in i ON t.txId = i.txId
LEFT JOIN linked_illicit l ON t.txId = l.txId

In [0]:
CREATE OR REPLACE TABLE gold.rule_flags AS
SELECT *,
  flag_fan_out * 30 + flag_fan_in * 30 + flag_linked_illicit * 40 AS risk_score
FROM all_flags

In [0]:
SELECT
  COUNT(*) AS total_rows,
  SUM(flag_fan_out) AS fan_out_flagged,
  SUM(flag_fan_in) AS fan_in_flagged,
  SUM(flag_linked_illicit) AS linked_flagged
FROM gold.rule_flags

%md
## Step 7: Do the rules work?
For each risk score, count how many transactions are illicit and how many are licit. If the rules work, higher scores should have more illicit transactions.

In [0]:
SELECT
  risk_score,
  COUNT(*) AS transactions,
  COUNT(CASE WHEN label_name = 'illicit' THEN 1 END) AS illicit,
  COUNT(CASE WHEN label_name = 'licit' THEN 1 END) AS licit
FROM gold.rule_flags
GROUP BY risk_score
ORDER BY risk_score

%md
## Findings
- Sending to or receiving from many others (score 30) is not a good sign of crime. Most of these are clean.
- Being linked to a known illicit transaction (score 40) is the best rule. About 6 in 10 are illicit.
- The rules missed about 8 in 10 illicit transactions. So we need a smarter method: a machine learning model in 06.